In [1]:
import pandas as pd
from pathlib import Path
from typing import List, Optional
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from torch import nn
import torch
from torch.utils.data import DataLoader, TensorDataset, Dataset
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import re
from scipy.stats import skew, kurtosis

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

In [2]:
def merge_csv(
    folder_path: Optional[Path] = None, 
    file_paths: Optional[List[Path]] = None
) -> pd.DataFrame:
    """
    Merges CSV files from either a specified folder or a list of file paths.

    Args:
        folder_path: The path to the folder containing CSV files to merge.
        file_paths: A list of specific CSV file paths to merge.

    Returns:
        A single pandas DataFrame containing the merged data.

    Raises:
        ValueError: If neither or both `folder_path` and `file_paths` are provided,
                    or if no CSV files are found.
    """
    # 1. Input Validation (Guard Clauses)
    # This ensures the function is called correctly.
    if (folder_path is None and file_paths is None) or \
       (folder_path is not None and file_paths is not None):
        raise ValueError("Provide either 'folder_path' OR 'file_paths', not both/neither.")

    # 2. Determine the list of files based on the mode
    if folder_path:
        files = list(folder_path.glob("*.csv"))
    else:  # This block runs if file_paths is provided
        files = file_paths

    if not files:
        raise ValueError("No CSV files found to merge.")

    # 3. Efficient Merging Logic with label extraction
    df_list = []
    for file_path in files:
        df = pd.read_csv(file_path)
        
        # Extract label from filename
        filename = file_path.name if hasattr(file_path, 'name') else str(file_path).split('/')[-1]
        
        # Handle both esp32_csi_* and wifisignal_data_* formats
        if 'esp32_csi_' in filename:
            pattern = r'esp32_csi_([a-zA-Z]+)_\d{8}_\d{6}\.csv'
        else:
            pattern = r'wifisignal_data_([a-zA-Z]+)_\d{8}_\d{6}\.csv'
        
        match = re.search(pattern, filename)
        if match:
            activity_label = match.group(1)
            df['label'] = activity_label
            df_list.append(df)
        else:
            print(f"Warning: Could not extract label from filename {filename}")
    
    if not df_list:
        raise ValueError("No valid CSV files processed.")
        
    return pd.concat(df_list, ignore_index=True)


def extract_packet_temporal_features(df: pd.DataFrame, window_size: int = 32) -> pd.DataFrame:
    """
    WiFi-HAR bulletproof temporal feature extraction from packet data.
    
    Args:
        df: DataFrame with packet columns (pkt0, pkt1, ..., pkt59) and 'label'
        window_size: Number of consecutive rows to use for temporal features
        
    Returns:
        DataFrame with temporal features extracted for each window
    """
    features_list = []
    
    # Get packet columns (excluding label)
    packet_cols = [col for col in df.columns if col.startswith('pkt')]
    
    if not packet_cols:
        raise ValueError("No packet columns found. Expected columns like 'pkt0', 'pkt1', etc.")
    
    print(f"WiFi-HAR Temporal Processing: {len(df)} rows, {len(packet_cols)} packet features, window size {window_size}")
    
    # WiFi-HAR bulletproof step calculation using overlap
    overlap_ratio = globals().get('TEMPORAL_OVERLAP', 0.75)
    
    # Bulletproof step size calculation - multiple guards
    if 0 <= overlap_ratio < 1:
        step_size = max(1, round(window_size * (1 - overlap_ratio)))
    else:
        print(f"Warning: Invalid overlap ratio {overlap_ratio}, using default step size")
        step_size = max(1, window_size // 4)  # Safe fallback
    
    # Additional safety - get from global config with fallbacks
    step_size = max(1, globals().get('TEMPORAL_STEP_SIZE', step_size))
    
    print(f"WiFi-HAR Windowing: overlap={overlap_ratio}, step_size={step_size}")
    
    # WiFi-HAR validation - ensure safe parameters
    if window_size <= 0:
        raise ValueError(f"Window size must be positive, got {window_size}")
    if step_size <= 0:
        raise ValueError(f"Step size must be positive, got {step_size}")
    if window_size > len(df):
        print(f"Warning: Window size {window_size} > data length {len(df)}, adjusting to {len(df)//2}")
        window_size = max(8, len(df) // 2)  # Ensure minimum window
        step_size = max(1, window_size // 4)
    
    # Calculate number of windows - bulletproof
    max_start_idx = max(0, len(df) - window_size)
    num_windows = max(1, (max_start_idx // step_size) + 1)
    
    print(f"WiFi-HAR Window Configuration:")
    print(f"  Data length: {len(df)}")
    print(f"  Window size: {window_size}")  
    print(f"  Step size: {step_size}")
    print(f"  Max start index: {max_start_idx}")
    print(f"  Expected windows: {num_windows}")
    
    # WiFi-HAR bulletproof windowing loop
    for i in range(0, max_start_idx + 1, step_size):
        if i % 1000 == 0:  # Progress indicator
            print(f"  Processing window {len(features_list)+1}/{num_windows} (index {i})")
        
        # Ensure we don't exceed data bounds
        end_idx = min(i + window_size, len(df))
        actual_window_size = end_idx - i
        
        # Skip windows that are too small
        if actual_window_size < max(4, window_size // 2):
            print(f"  Skipping window at {i}: too small ({actual_window_size} < {window_size//2})")
            continue
        
        # Extract window of data
        window_data = df.iloc[i:end_idx]
        packet_data = window_data[packet_cols].values  # Shape: (actual_window_size, n_packets)
        
        try:
            # Compute temporal features for this window
            features = compute_packet_temporal_features(packet_data)
            features['label'] = window_data['label'].iloc[-1]  # Use last label in window
            features['window_size'] = actual_window_size  # Track actual window size
            features_list.append(features)
        except Exception as e:
            print(f"Warning: Failed to compute features for window {i}: {e}")
            continue
    
    print(f"WiFi-HAR Extraction Complete: {len(features_list)} feature windows extracted")
    
    if not features_list:
        raise ValueError("No features extracted! Check window size and data length.")
    
    return pd.DataFrame(features_list)


def compute_packet_temporal_features(packet_data: np.ndarray) -> dict:
    """
    Compute enhanced temporal movement features from packet data.
    
    Args:
        packet_data: Array of shape (time_steps, n_packets) containing packet values
        
    Returns:
        Dictionary of temporal features
    """
    features = {}
    
    # Handle NaN values
    packet_data = np.nan_to_num(packet_data, nan=0.0)
    
    # 1. Enhanced temporal derivatives (movement indicators)
    temporal_diffs = np.diff(packet_data, axis=0)  # First-order differences
    temporal_diffs2 = np.diff(temporal_diffs, axis=0)  # Second-order differences (acceleration)
    temporal_diffs3 = np.diff(temporal_diffs2, axis=0) if temporal_diffs2.shape[0] > 1 else np.array([[]])  # Third-order (jerk)
    
    # 2. Enhanced per-packet temporal features
    packet_means = np.mean(temporal_diffs, axis=0)
    packet_stds = np.std(temporal_diffs, axis=0)
    packet_maxs = np.max(np.abs(temporal_diffs), axis=0)
    packet_mins = np.min(temporal_diffs, axis=0)
    packet_ranges = np.ptp(packet_data, axis=0)
    packet_vars = np.var(packet_data, axis=0)
    packet_medians = np.median(packet_data, axis=0)
    
    # 3. Enhanced global temporal features
    global_temporal_mean = np.mean(temporal_diffs)
    global_temporal_std = np.std(temporal_diffs)
    global_temporal_max = np.max(np.abs(temporal_diffs))
    global_temporal_min = np.min(temporal_diffs)
    global_temporal_energy = np.sum(temporal_diffs**2)
    global_temporal_rms = np.sqrt(np.mean(temporal_diffs**2))
    
    # 4. Enhanced acceleration features (second-order)
    if temporal_diffs2.size > 0:
        global_accel_mean = np.mean(temporal_diffs2)
        global_accel_std = np.std(temporal_diffs2)
        global_accel_max = np.max(np.abs(temporal_diffs2))
        global_accel_energy = np.sum(temporal_diffs2**2)
    else:
        global_accel_mean = global_accel_std = global_accel_max = global_accel_energy = 0.0
    
    # 5. Jerk features (third-order) for activity smoothness
    if temporal_diffs3.size > 0:
        global_jerk_mean = np.mean(temporal_diffs3)
        global_jerk_std = np.std(temporal_diffs3)
        global_jerk_max = np.max(np.abs(temporal_diffs3))
    else:
        global_jerk_mean = global_jerk_std = global_jerk_max = 0.0
    
    # 6. Enhanced cross-packet correlations
    n_corr_samples = globals().get('N_CORRELATION_SAMPLES', 15)
    indices = np.linspace(0, packet_data.shape[1]-1, min(n_corr_samples, packet_data.shape[1]), dtype=int)
    sampled_data = packet_data[:, indices]
    
    try:
        if sampled_data.shape[1] > 1:
            corr_matrix = np.corrcoef(sampled_data.T)
            corr_matrix = np.nan_to_num(corr_matrix, nan=0.0)
            upper_tri = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
            
            corr_mean = np.mean(upper_tri) if len(upper_tri) > 0 else 0.0
            corr_std = np.std(upper_tri) if len(upper_tri) > 0 else 0.0
            corr_max = np.max(np.abs(upper_tri)) if len(upper_tri) > 0 else 0.0
            corr_min = np.min(upper_tri) if len(upper_tri) > 0 else 0.0
        else:
            corr_mean = corr_std = corr_max = corr_min = 0.0
    except:
        corr_mean = corr_std = corr_max = corr_min = 0.0
    
    # 7. Enhanced statistical features
    try:
        temporal_skewness = skew(temporal_diffs.flatten()) if temporal_diffs.size > 1 else 0.0
        temporal_kurtosis = kurtosis(temporal_diffs.flatten()) if temporal_diffs.size > 1 else 0.0
        # Add percentile features
        temporal_p25 = np.percentile(temporal_diffs.flatten(), 25) if temporal_diffs.size > 0 else 0.0
        temporal_p75 = np.percentile(temporal_diffs.flatten(), 75) if temporal_diffs.size > 0 else 0.0
        temporal_iqr = temporal_p75 - temporal_p25
    except:
        temporal_skewness = temporal_kurtosis = temporal_p25 = temporal_p75 = temporal_iqr = 0.0
    
    # 8. Enhanced frequency domain features
    try:
        # Compute variance across packets for each time step
        time_variances = np.var(packet_data, axis=1)
        variance_trend = np.corrcoef(np.arange(len(time_variances)), time_variances)[0, 1] if len(time_variances) > 1 else 0.0
        
        # Activity pattern features
        mean_variance = np.mean(time_variances)
        std_variance = np.std(time_variances)
        
        # Temporal consistency
        consistency_score = 1.0 / (1.0 + std_variance) if std_variance > 0 else 1.0
    except:
        variance_trend = mean_variance = std_variance = consistency_score = 0.0
    
    # 9. Enhanced activity signature features
    movement_intensity = np.sqrt(global_temporal_energy)
    activity_smoothness = global_temporal_std / (np.abs(global_temporal_mean) + 1e-8)
    peak_activity = global_temporal_max
    
    # Activity type indicators
    is_static = 1.0 if movement_intensity < 0.1 else 0.0
    is_dynamic = 1.0 if movement_intensity > 1.0 else 0.0
    is_periodic = 1.0 if 0.3 < consistency_score < 0.8 else 0.0
    
    # Collect enhanced features
    n_packet_features = globals().get('N_PACKET_FEATURES', 30)
    features.update({
        # Enhanced per-packet temporal features
        **{f'temporal_mean_{i}': packet_means[i] for i in range(min(n_packet_features, len(packet_means)))},
        **{f'temporal_std_{i}': packet_stds[i] for i in range(min(n_packet_features, len(packet_stds)))},
        **{f'temporal_max_{i}': packet_maxs[i] for i in range(min(n_packet_features, len(packet_maxs)))},
        **{f'temporal_min_{i}': packet_mins[i] for i in range(min(n_packet_features, len(packet_mins)))},
        **{f'packet_var_{i}': packet_vars[i] for i in range(min(n_packet_features, len(packet_vars)))},
        **{f'packet_range_{i}': packet_ranges[i] for i in range(min(n_packet_features, len(packet_ranges)))},
        **{f'packet_median_{i}': packet_medians[i] for i in range(min(n_packet_features, len(packet_medians)))},
        
        # Enhanced global temporal features
        'global_temporal_mean': global_temporal_mean,
        'global_temporal_std': global_temporal_std,
        'global_temporal_max': global_temporal_max,
        'global_temporal_min': global_temporal_min,
        'global_temporal_energy': global_temporal_energy,
        'global_temporal_rms': global_temporal_rms,
        
        # Enhanced acceleration features
        'global_accel_mean': global_accel_mean,
        'global_accel_std': global_accel_std,
        'global_accel_max': global_accel_max,
        'global_accel_energy': global_accel_energy,
        
        # Jerk features
        'global_jerk_mean': global_jerk_mean,
        'global_jerk_std': global_jerk_std,
        'global_jerk_max': global_jerk_max,
        
        # Enhanced cross-packet correlation features
        'packet_corr_mean': corr_mean,
        'packet_corr_std': corr_std,
        'packet_corr_max': corr_max,
        'packet_corr_min': corr_min,
        
        # Enhanced statistical features
        'temporal_skewness': temporal_skewness,
        'temporal_kurtosis': temporal_kurtosis,
        'temporal_p25': temporal_p25,
        'temporal_p75': temporal_p75,
        'temporal_iqr': temporal_iqr,
        
        # Enhanced trend features
        'variance_trend': variance_trend,
        'mean_variance': mean_variance,
        'std_variance': std_variance,
        'consistency_score': consistency_score,
        
        # Enhanced activity signature features
        'movement_intensity': movement_intensity,
        'activity_smoothness': activity_smoothness,
        'peak_activity': peak_activity,
        'is_static': is_static,
        'is_dynamic': is_dynamic,
        'is_periodic': is_periodic,
    })
    
    # Clean any NaN or infinite values
    for key, value in features.items():
        if np.isnan(value) or np.isinf(value):
            features[key] = 0.0
    
    return features


def preprocess(df : pd.DataFrame, target_column = 'label', n_components = 0.99, batch_size = 256):
    feature_columns = [col for col in df.columns if col != target_column]
    
    le = ColumnTransformer(
        transformers = [
            ("encoder", OrdinalEncoder(), [target_column])
        ],
        remainder = "passthrough",
        verbose_feature_names_out = False
    )
    
    scaler = ColumnTransformer(
        transformers = [
            ("scaler", StandardScaler(), feature_columns)
        ],
        remainder = "passthrough",
        verbose_feature_names_out = False
    )
    
    pca = ColumnTransformer(
        transformers = [
            ("pca", PCA(), feature_columns)
        ],
        remainder = "passthrough",
        verbose_feature_names_out = False
    )
    
    pipeline = make_pipeline(le, scaler, pca)
    
    pipeline.set_output(transform = "pandas")

    df = pipeline.fit_transform(df)


    X = df.drop(target_column, axis=1).to_numpy()
    y = df[target_column].to_numpy()
    
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.long)

    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
    
    return loader



class PreprocessedDataset(Dataset):
    def __init__(self, df: pd.DataFrame, target_column='label', n_components=0.99, pipeline = None, use_temporal=True):
        # Extract temporal features if enabled
        if use_temporal:
            print("Extracting enhanced temporal movement features from packet data...")
            # Use the improved window size from global config
            window_size = globals().get('DOPPLER_WINDOW_SIZE', 32)
            df = extract_packet_temporal_features(df, window_size=window_size)
            print(f"Enhanced temporal features extracted. Shape: {df.shape}")
        
        feature_columns = [col for col in df.columns if col != target_column]

        if pipeline is None:
            # Step 1: Encode target
            le = ColumnTransformer(
                transformers=[
                    ("encoder", OrdinalEncoder(), [target_column])
                ],
                remainder="passthrough",
                verbose_feature_names_out=False
            )
    
            # Step 2: Scale features
            scaler = ColumnTransformer(
                transformers=[
                    ("scaler", StandardScaler(), feature_columns)
                ],
                remainder="passthrough",
                verbose_feature_names_out=False
            )
    
            # Step 3: PCA for features
            pca = ColumnTransformer(
                transformers=[
                    ("pca", PCA(n_components=n_components), feature_columns)
                ],
                remainder="passthrough",
                verbose_feature_names_out=False
            )
    
            # Step 4: Create pipeline
            pipeline = Pipeline(
                steps = [
                    ('le', le),
                    ('scaler', scaler),
                    ('pca', pca)
                ]
            )
            pipeline.set_output(transform="pandas")
        
            
            # Step 5: Apply preprocessing
            df_transformed = pipeline.fit_transform(df)
            
        else:
            df_transformed = pipeline.transform(df)
            le = pipeline.named_steps['le']
            scaler = pipeline.named_steps['scaler']
            pca= pipeline.named_steps['pca']

        # Step 6: Save as tensors
        self.X = torch.tensor(
            df_transformed.drop(target_column, axis=1).to_numpy(),
            dtype=torch.float32
        )
        self.y = torch.tensor(
            df_transformed[target_column].to_numpy(),
            dtype=torch.long
        )

        self.le_ = le
        self.scaler_ = scaler
        self.pca_ = pca
        self.pipeline_ = pipeline

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [3]:
class AggressiveModel(nn.Module):
    def __init__(self, input_size, num_classes, dropout_rate=0.1):
        super(AggressiveModel, self).__init__()
        
        self.input_size = input_size
        
        # Much wider and simpler architecture for raw packet data
        self.fc1 = nn.Linear(input_size, 1024)  # Wider first layer
        self.bn1 = nn.BatchNorm1d(1024)
        self.dropout1 = nn.Dropout(dropout_rate)  # Very low dropout
        
        self.fc2 = nn.Linear(1024, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.fc_out = nn.Linear(512, num_classes)  # Direct to output
        
        # Initialize weights aggressively
        self._init_weights()
    
    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.BatchNorm1d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)
    
    def forward(self, x):
        # Simple and wide network
        x = self.fc1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.dropout2(x)
        
        x = self.fc_out(x)
        return x

In [4]:
DATA_PATH = Path('.')
SEED = 42
BATCH_SIZE = 256
PCA_COMPONENTS = 0.8  # Much less compression - keep 80% instead of 95%
TARGET_COLUMN = 'label'

# Dramatically simplified temporal parameters 
DOPPLER_WINDOW_SIZE = 16  # Smaller window for more samples
TEMPORAL_OVERLAP = 0.25   # Much less overlap for more diverse training data
TEMPORAL_STEP_SIZE = max(1, round(DOPPLER_WINDOW_SIZE * (1 - TEMPORAL_OVERLAP)))
USE_TEMPORAL = False      # DISABLE temporal features initially - raw data only

print(f"AGGRESSIVE Parameter Changes:")
print(f"  Window size: {DOPPLER_WINDOW_SIZE}")
print(f"  Overlap ratio: {TEMPORAL_OVERLAP}")
print(f"  Step size: {TEMPORAL_STEP_SIZE}")
print(f"  Using temporal features: {USE_TEMPORAL}")

# Minimal feature extraction
N_CORRELATION_SAMPLES = 5   # Minimal complexity
N_PACKET_FEATURES = 10      # Much reduced

N_TRAIN = 4
N_VALID = 1

EPOCHS = 100  # More epochs since we have simpler model
EARLY_STOPPING_PATIENCE = 15  # More patience
MODEL_PATH = Path("modelcheckpoints/best_model_aggressive.pt")

# AGGRESSIVE learning parameters
LEARNING_RATE = 5e-3   # 5x higher learning rate
WEIGHT_DECAY = 5e-5    # Much less regularization
GRADIENT_CLIP = 5.0    # Higher gradient clipping

rng = np.random.default_rng(seed = SEED)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"\nUsing device: {device}")
print(f"AGGRESSIVE parameters:")
print(f"  Window size: {DOPPLER_WINDOW_SIZE}")
print(f"  Step size: {TEMPORAL_STEP_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  PCA components: {PCA_COMPONENTS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Use temporal: {USE_TEMPORAL}")

AGGRESSIVE Parameter Changes:
  Window size: 16
  Overlap ratio: 0.25
  Step size: 12
  Using temporal features: False

Using device: mps
AGGRESSIVE parameters:
  Window size: 16
  Step size: 12
  Batch size: 256
  PCA components: 0.8
  Learning rate: 0.005
  Weight decay: 5e-05
  Use temporal: False


In [5]:
paths = np.array(list(DATA_PATH.glob('wifisignal_data_*.csv')))
pattern = r'wifisignal_data_([a-zA-Z]+)_(\d{8})_(\d{6})\.csv'

labels = []
datetimes = []

for path in paths:
    match = re.search(pattern, path.name)
    if match:
        label = match.group(1)
        date_str = match.group(2)
        time_str = match.group(3)
        
        # Combine into full datetime string
        datetime_str = date_str + time_str  
        
        # Convert to pandas datetime
        dt = pd.to_datetime(datetime_str, format="%Y%m%d%H%M%S")
        
        labels.append(label)
        datetimes.append(dt)

paths_df = pd.DataFrame(
    data = {
        'path' : paths,
        'label' : labels,
        'datetime' : datetimes
    }
)

# Extract date for filtering  
paths_df['date'] = paths_df['datetime'].dt.strftime('%Y%m%d')

# AGGRESSIVE DATA STRATEGY: Use MORE training data
train_data = paths_df[paths_df['date'] == '20250528']
train_paths = []

print("AGGRESSIVE Training Strategy - More data per activity:")
for activity in train_data['label'].unique():
    activity_files = train_data[train_data['label'] == activity].sort_values('datetime')
    # Take ALL files for each activity (not just 2)
    selected_files = activity_files['path'].tolist()
    train_paths.extend(selected_files)
    print(f"{activity}: {len(selected_files)} files")
    for path in selected_files:
        print(f"  {path.name}")

# Select test files: LATEST file per activity from ALL dates
test_paths = []

print(f"\nTest files (latest file per activity from all dates):")
for activity in paths_df['label'].unique():
    activity_files = paths_df[paths_df['label'] == activity].sort_values('datetime')
    # Take LAST file for each activity (latest timestamp)
    latest_file = activity_files.tail(1)['path'].tolist()
    test_paths.extend(latest_file)
    print(f"{activity}: {len(latest_file)} files")
    for path in latest_file:
        print(f"  {path.name}")

print(f"\nTotal: {len(train_paths)} training files, {len(test_paths)} test files")

# Ensure we have both training and test data
if not train_paths:
    raise ValueError("No training data found")
if not test_paths:
    raise ValueError("No test data found")
if len(test_paths) != 5:
    print(f"Warning: Expected 5 test files (one per activity), but found {len(test_paths)}")

# Load and preprocess data with AGGRESSIVE settings
print("\n" + "="*70)
print("LOADING DATA WITH AGGRESSIVE SETTINGS - RAW DATA FOCUS")
print("="*70)

train_df = merge_csv(file_paths=train_paths)
print(f"Raw training data shape: {train_df.shape}")
print(f"Available columns: {train_df.columns.tolist()}")

train_ds = PreprocessedDataset(
    train_df, 
    target_column=TARGET_COLUMN, 
    n_components=PCA_COMPONENTS, 
    use_temporal=USE_TEMPORAL  # False - using raw data
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
X_train, y_train = train_ds.X, train_ds.y

test_df = merge_csv(file_paths=test_paths)
print(f"Raw test data shape: {test_df.shape}")

test_ds = PreprocessedDataset(
    test_df, 
    target_column=TARGET_COLUMN, 
    n_components=PCA_COMPONENTS, 
    pipeline=train_ds.pipeline_,
    use_temporal=USE_TEMPORAL  # False - using raw data
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
X_test, y_test = test_ds.X, test_ds.y

le = train_ds.le_

print(f"\nFinal processed shapes:")
print(f"Training: X={X_train.shape}, y={y_train.shape}")
print(f"Test: X={X_test.shape}, y={y_test.shape}")
print(f"Activity classes: {le.named_transformers_['encoder'].categories_[0]}")

# Validate data loader lengths
if len(train_loader) == 0:
    raise ValueError("Training data loader is empty.")
if len(test_loader) == 0:
    raise ValueError("Test data loader is empty.")

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

# Initialize AGGRESSIVE model
model = AggressiveModel(
    input_size=X_train.shape[1], 
    num_classes=len(le.named_transformers_["encoder"].categories_[0]),
    dropout_rate=0.1  # Very low dropout
)
model = model.to(device)

print(f"\nAGGRESSIVE Model Configuration:")
print(f"Input size: {X_train.shape[1]}")
print(f"Number of classes: {len(le.named_transformers_['encoder'].categories_[0])}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# AGGRESSIVE training setup - no class weights initially
print(f"\nAGGRESSIVE Training Setup - Unweighted loss:")

# No class weighting to start
criterion = nn.CrossEntropyLoss()

# AGGRESSIVE optimizer
optimizer = torch.optim.SGD(  # SGD can be more aggressive than Adam
    model.parameters(), 
    lr=LEARNING_RATE, 
    momentum=0.9,
    weight_decay=WEIGHT_DECAY
)

# Aggressive learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=EPOCHS,
    eta_min=1e-6
)

print(f"AGGRESSIVE Training Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Optimizer: SGD with momentum")

# Training metrics storage
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []
learning_rates = []
epochs_completed = []

# Early stopping variables
patience = EARLY_STOPPING_PATIENCE
best_test_loss = float('inf')
best_test_acc = 0.0

print(f"\n" + "="*70)
print("STARTING AGGRESSIVE TRAINING")
print("="*70)

# AGGRESSIVE training loop with data augmentation
for epoch in range(EPOCHS):
    # Training phase
    model.train()
    running_loss = 0.0
    train_correct, train_total = 0, 0
    
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    for batch_idx, (xb, yb) in enumerate(progress):
        xb, yb = xb.to(device), yb.to(device)
        
        # Simple data augmentation: add small noise
        if model.training:
            noise = torch.randn_like(xb) * 0.01
            xb = xb + noise
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        
        # Aggressive gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
        
        optimizer.step()
        
        running_loss += loss.item()
        train_correct += (preds.argmax(dim=1) == yb).sum().item()
        train_total += yb.size(0)
        
        current_lr = scheduler.get_last_lr()[0]
        progress.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Acc': f'{100*train_correct/train_total:.1f}%',
            'LR': f'{current_lr:.2e}'
        })

    train_loss = running_loss / len(train_loader)
    train_acc = train_correct / train_total
    
    # Update learning rate
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    # Test phase
    model.eval()
    test_loss, test_correct, test_total = 0.0, 0, 0

    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            test_loss += loss.item()
            test_correct += (preds.argmax(dim=1) == yb).sum().item()
            test_total += yb.size(0)

    test_loss /= len(test_loader)
    test_acc = test_correct / test_total

    # Store metrics
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)
    learning_rates.append(current_lr)
    epochs_completed.append(epoch + 1)

    print(f"Epoch {epoch+1:3d}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
          f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}, LR: {current_lr:.2e}")

    # Early stopping logic
    if test_acc > best_test_acc:  # Focus on accuracy
        best_test_loss = test_loss
        best_test_acc = test_acc
        
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'epoch': epoch + 1,
            'best_test_loss': best_test_loss,
            'best_test_acc': best_test_acc,
            'train_losses': train_losses,
            'test_losses': test_losses,
            'train_accuracies': train_accuracies,
            'test_accuracies': test_accuracies
        }, MODEL_PATH)
        
        patience = EARLY_STOPPING_PATIENCE
        print(f"  → NEW BEST ACC: {best_test_acc:.4f} - Model saved")
    else:
        patience -= 1
        print(f"  → No improvement. Patience: {patience}")
        if patience == 0:
            print("  → Early stopping triggered.")
            break

print(f"\n" + "="*70)
print(f"AGGRESSIVE TRAINING COMPLETED!")
print(f"Best test loss: {best_test_loss:.4f}")
print(f"Best test accuracy: {best_test_acc:.4f}")
if test_accuracies:
    print(f"Final test accuracy: {test_accuracies[-1]:.4f}")
    improvement = best_test_acc - test_accuracies[0]
    print(f"Improvement: {improvement:.4f} ({improvement*100:.1f}%)")
print("="*70)

AGGRESSIVE Training Strategy - More data per activity:
falling: 30 files
  wifisignal_data_falling_20250528_175224.csv
  wifisignal_data_falling_20250528_175355.csv
  wifisignal_data_falling_20250528_175522.csv
  wifisignal_data_falling_20250528_175649.csv
  wifisignal_data_falling_20250528_175813.csv
  wifisignal_data_falling_20250528_180009.csv
  wifisignal_data_falling_20250528_180132.csv
  wifisignal_data_falling_20250528_180315.csv
  wifisignal_data_falling_20250528_180453.csv
  wifisignal_data_falling_20250528_180631.csv
  wifisignal_data_falling_20250528_180857.csv
  wifisignal_data_falling_20250528_181050.csv
  wifisignal_data_falling_20250528_181215.csv
  wifisignal_data_falling_20250528_181400.csv
  wifisignal_data_falling_20250528_181608.csv
  wifisignal_data_falling_20250528_181801.csv
  wifisignal_data_falling_20250528_181944.csv
  wifisignal_data_falling_20250528_182115.csv
  wifisignal_data_falling_20250528_182244.csv
  wifisignal_data_falling_20250528_182420.csv
  wifis

Epoch   1, Train Loss: 1.7121, Train Acc: 0.3298, Test Loss: 1.3871, Test Acc: 0.3520, LR: 5.00e-03
  → NEW BEST ACC: 0.3520 - Model saved


Epoch   2, Train Loss: 1.4022, Train Acc: 0.3659, Test Loss: 1.3767, Test Acc: 0.3561, LR: 5.00e-03
  → NEW BEST ACC: 0.3561 - Model saved


Epoch   3, Train Loss: 1.3786, Train Acc: 0.3780, Test Loss: 1.4552, Test Acc: 0.3226, LR: 4.99e-03
  → No improvement. Patience: 14


Epoch   4, Train Loss: 1.3619, Train Acc: 0.3858, Test Loss: 1.4150, Test Acc: 0.3245, LR: 4.98e-03
  → No improvement. Patience: 13


...

Epoch  17, Train Loss: 1.2676, Train Acc: 0.4335, Test Loss: 1.5078, Test Acc: 0.3267, LR: 4.65e-03
  → No improvement. Patience: 14


Epoch  18, Train Loss: 1.2630, Train Acc: 0.4356, Test Loss: 1.4531, Test Acc: 0.3629, LR: 4.61e-03
  → No improvement. Patience: 13


Epoch  19, Train Loss: 1.2606, Train Acc: 0.4368, Test Loss: 1.4822, Test Acc: 0.3482, LR: 4.57e-03
  → No improvement. Patience: 12


Epoch  20, Train Loss: 1.2570, Train Acc: 0.4386, Test Loss: 1.4505, Test Acc: 0.3867, LR: 4.52e-03
  → No improvement. Patience: 11


KeyboardInterrupt: 

In [ ]:
model.eval()

torch.save(model.state_dict(), "modelcheckpoints/notTrainTestSplit.pt")

In [ ]:
# Create comprehensive training metrics visualization
plt.figure(figsize=(15, 10))

# 1. Loss curves
plt.subplot(2, 3, 1)
plt.plot(epochs_completed, train_losses, 'b-', label='Training Loss', linewidth=2)
plt.plot(epochs_completed, test_losses, 'r-', label='Test Loss', linewidth=2)
plt.title('Training & Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Accuracy curves
plt.subplot(2, 3, 2)
plt.plot(epochs_completed, train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
plt.plot(epochs_completed, test_accuracies, 'r-', label='Test Accuracy', linewidth=2)
plt.title('Training & Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# 3. Training vs Test Loss (scatter plot)
plt.subplot(2, 3, 3)
plt.scatter(train_losses, test_losses, c=epochs_completed, cmap='viridis', s=50)
plt.plot([min(train_losses), max(train_losses)], [min(train_losses), max(train_losses)], 'k--', alpha=0.5)
plt.title('Training vs Test Loss')
plt.xlabel('Training Loss')
plt.ylabel('Test Loss')
plt.colorbar(label='Epoch')
plt.grid(True, alpha=0.3)

# 4. Learning curves (combined)
plt.subplot(2, 3, 4)
ax1 = plt.gca()
ax1.plot(epochs_completed, train_losses, 'b-', linewidth=2, label='Train Loss')
ax1.plot(epochs_completed, test_losses, 'r-', linewidth=2, label='Test Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss', color='k')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(epochs_completed, train_accuracies, 'b--', linewidth=2, label='Train Acc')
ax2.plot(epochs_completed, test_accuracies, 'r--', linewidth=2, label='Test Acc')
ax2.set_ylabel('Accuracy', color='k')
ax2.legend(loc='upper right')
plt.title('Combined Learning Curves')

# 5. Overfitting analysis
plt.subplot(2, 3, 5)
gap = np.array(train_accuracies) - np.array(test_accuracies)
plt.plot(epochs_completed, gap, 'g-', linewidth=2, label='Train-Test Accuracy Gap')
plt.axhline(y=0, color='k', linestyle='--', alpha=0.5)
plt.title('Overfitting Analysis')
plt.xlabel('Epoch')
plt.ylabel('Accuracy Gap (Train - Test)')
plt.legend()
plt.grid(True, alpha=0.3)

# 6. Metrics summary table
plt.subplot(2, 3, 6)
plt.axis('off')

# Create summary statistics
final_train_loss = train_losses[-1]
final_test_loss = test_losses[-1]
final_train_acc = train_accuracies[-1]
final_test_acc = test_accuracies[-1]
best_test_acc = max(test_accuracies)
best_test_acc_epoch = epochs_completed[np.argmax(test_accuracies)]

summary_text = f"""
Training Summary:

Final Training Loss: {final_train_loss:.4f}
Final Test Loss: {final_test_loss:.4f}
Final Training Accuracy: {final_train_acc:.4f}
Final Test Accuracy: {final_test_acc:.4f}

Best Test Accuracy: {best_test_acc:.4f}
Best Test Acc at Epoch: {best_test_acc_epoch}

Total Epochs Completed: {len(epochs_completed)}
Early Stopping: {'Yes' if len(epochs_completed) < EPOCHS else 'No'}

Overfitting Gap: {final_train_acc - final_test_acc:.4f}
"""

plt.text(0.1, 0.9, summary_text, transform=plt.gca().transAxes, 
         fontsize=10, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.5))

plt.tight_layout()
plt.show()

# Print detailed metrics
print("=== Detailed Training Metrics ===")
print(f"{'Epoch':<6} {'Train Loss':<12} {'Train Acc':<12} {'Test Loss':<12} {'Test Acc':<12} {'Gap':<8}")
print("-" * 68)
for i in range(len(epochs_completed)):
    gap = train_accuracies[i] - test_accuracies[i]
    print(f"{epochs_completed[i]:<6} {train_losses[i]:<12.4f} {train_accuracies[i]:<12.4f} "
          f"{test_losses[i]:<12.4f} {test_accuracies[i]:<12.4f} {gap:<8.4f}")

In [ ]:
model.to(device)

categories = le.named_transformers_['encoder'].categories_[0]

cm = np.zeros((5,5))

for X_batch, y_batch in tqdm(train_loader):
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    pred = model(X_batch).argmax(dim = 1)
    pred = pred.to('cpu')
    y_batch = y_batch.to('cpu')
    cm += confusion_matrix(y_batch, pred, labels = np.arange(len(categories)))

fig, ax = plt.subplots()
cm_df = pd.DataFrame(cm, columns = categories, index = categories, dtype = int)

ax.set_title("Confusion Matrix on Training Data")

sns.heatmap(cm_df, cmap = "Blues", annot = True, fmt = 'd', ax = ax)

ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")

plt.tight_layout()

plt.show()

In [ ]:
model.to(device)

categories = le.named_transformers_['encoder'].categories_[0]

cm = np.zeros((5,5))

for X_batch, y_batch in tqdm(test_loader):
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    pred = model(X_batch).argmax(dim = 1)
    pred = pred.to('cpu')
    y_batch = y_batch.to('cpu')
    cm += confusion_matrix(y_batch, pred, labels = np.arange(len(categories)))

fig, ax = plt.subplots()
cm_df = pd.DataFrame(cm, columns = categories, index = categories, dtype = int)

ax.set_title("Confusion Matrix on Test Data")

sns.heatmap(cm_df, cmap = "Blues", annot = True, fmt = 'd', ax = ax)

ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")

plt.tight_layout()

plt.show()